# CDF and Community Projects Dataset - Kalomo Town Council

**Owner:** Louis | **CSC4792 Mini Project, Group 44 - Kalomo Town Council, Zambia**

I used scanned Constituency Development Fund (CDF) project lists from Kalomo Town Council to produce `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv`. This notebook shows how I selected the source files, extracted the text, cleaned the results, and checked the final dataset.

The supporting scripts are:

- `scripts/scraping/scrape_cdf_projects.py`
- `scripts/cleaning/clean_cdf_projects.py`

I ran the OCR stage in Google Colab so EasyOCR could use a GPU. This notebook works with the saved raw and processed CSV files, so the PDFs and OCR step are not needed each time it is opened.

## Step 1 - Try scraping the council pages

I first ran automated scraping and cleaning scripts against the council's CDF pages. The scraper could collect only the general information shown on those pages and did not reach the detailed project records. After cleaning, this first attempt produced nine rows of basic information, which was not enough for the project-level dataset. That output was only an initial test and is not used in the final file.

## Step 2 - Find and select the source documents

I then visited both constituency pages myself to look for more detailed records. I found 20 downloadable PDFs: 10 under Kalomo Central and 10 under Dundumwezi. Each constituency page contained four community-project files, three skills-development bursary files, and three boarding-school bursary files.

From the eight community-project files, I selected these four:

1. `CDF-DUNDUMWEZI-2024.pdf`
2. `CDF-KALOMO-CENTRAL-2024.pdf`
3. `2025-DUNDUMWEZI-APPROVED-AND-NOT-APPROVED-PROJECTS.pdf`
4. `2025-KALOMO-CENTRAL-NOT-APPROVED-AND-APPROVED-PROJECTS.pdf`

For each constituency, I used the 2024 project list and the 2025 combined approved and not-approved list. I left out the separate 2025 project and not-approved files because their records are already covered by the combined lists. The 12 bursary files were also excluded because they contain people rather than community projects and do not fit the agreed CDF project columns.

In [ ]:
from pathlib import Path
import sys

repo_root = Path("..").resolve()
if not (repo_root / "scripts").exists():
    repo_root = Path(".").resolve()
sys.path.insert(0, str(repo_root / "scripts" / "scraping"))

import scrape_cdf_projects as scraper

print("Project PDFs used:")
for filename in scraper.CANONICAL_PROJECT_PDFS:
    print(" -", filename)

## Step 3 - Test direct PDF extraction

I first tried to extract the PDF tables with `pdfplumber`. It could not return usable table text because the pages were scanned images rather than PDFs with selectable text. This is why I moved to OCR for the next attempt.

## Step 4 - Extract the scanned tables with OCR

The selected PDFs are scans, so normal PDF text extraction could not read the tables properly. `scrape_cdf_projects.py` renders each page at 300 DPI with `pypdfium2`, then passes the image to EasyOCR.

EasyOCR returns the detected text, a confidence score, and the position of each text box. I kept the coordinates because each detection is only a table cell or text fragment, not a complete project row. The raw file contains:

- `text_line`
- `source_file_name`
- `page_number`
- `x_min`, `y_min`, `x_max`, `y_max`
- `ocr_confidence`

The OCR command used in Colab was:

```bash
python scripts/scraping/scrape_cdf_projects.py --pdf-dir /content/cdf_pdfs --gpu
```

This produced 3,804 OCR fragments in `data/raw/cdf_projects/raw_cdf_projects.csv`. That figure counts text boxes, not projects.

In [ ]:
import pandas as pd

raw_path = repo_root / "data" / "raw" / "cdf_projects" / "raw_cdf_projects.csv"
raw = pd.read_csv(raw_path)

print("Raw OCR fragments:", len(raw))
print("Source PDFs:", raw["source_file_name"].nunique())
display(raw.head())
display(raw.groupby(["source_file_name", "page_number"]).size().rename("fragments").to_frame())

## Step 5 - Problems encountered

**`pdfplumber` could not read the tables.** The PDFs looked like normal tables, but their pages were images and did not contain usable text for direct extraction. This led me to use OCR instead.

**The first OCR file had about 28,000 rows.** Each OCR text box was saved as a row, and the run also included thousands of bursary entries. The figure was a count of text fragments, not a count of projects.

**Some of the PDFs repeat the same projects.** The council publishes combined lists as well as separate approved and not-approved lists. I limited the scraper to the four complete files in Step 2 to avoid counting the same application twice.

**Reading the OCR output as plain text mixed up table rows.** Ward names, descriptions, and headings were sometimes treated as project names. I kept the text-box coordinates in the raw file so the cleaner could place each fragment back in the correct row and column.

**Some OCR text was misspelled.** Examples include `NOL APPROVED` instead of `NOT APPROVED`, `AIl Wards` instead of `All Wards`, and misspellings of `Infrastructure`. The cleaner matches these against known ward and sector names. If a value is still unclear, it remains `N/A`.

## Step 6 - Turn the OCR fragments into project records

The 2024 and 2025 tables have different layouts, so the cleaner handles them separately.

In the 2024 tables, each recognised sector marks a project row. The cleaner uses the nearby sector boxes to estimate the top and bottom of that row, then reads the project name and ward from the same area.

In the 2025 tables, each `Approved` or `Not Approved` entry marks a project row. The column headings show where to read the name, description, sector, type, ward, comments, and reason. For tightly packed rows, the serial numbers help separate one project name from the next.

The cleaner also:

- removes headings, row numbers, rejection reasons, stamps, and letterhead text from project names;
- standardises recognised wards, sectors, and statuses;
- keeps repeated applications when they are separate entries in the source;
- assigns IDs starting at `CDF-0001`;
- records the council source page and cleaning date;
- checks project names for NRC and phone-number patterns; and
- saves the final file with a pipe (`|`) delimiter.

The source tables do not give reliable allocation or disbursement amounts for each project, so both amount columns are left as `N/A`.

In [ ]:
sys.path.insert(0, str(repo_root / "scripts" / "cleaning"))
import clean_cdf_projects as cleaner

print("Final column order:")
for column in cleaner.OUTPUT_COLUMNS:
    print(" -", column)

print("\nTo rebuild the processed file without repeating OCR:")
print("python scripts/cleaning/clean_cdf_projects.py")

## Step 7 - Load and inspect the final dataset

In [ ]:
processed_path = repo_root / "data" / "processed" / "db-unza26-csc4792-kalomo_town_council_cdf_projects.csv"
df = pd.read_csv(
    processed_path, sep="|", keep_default_na=False, dtype={"fiscal_year": str}
)
df.head(10)

In [ ]:
print("Total project records:", len(df))
print("\nRecords by year and status:")
display(df.groupby(["fiscal_year", "status"]).size().rename("records").to_frame())

print("Records by sector:")
display(df["sector"].value_counts().rename("records").to_frame())

print("Records by ward:")
display(df["ward"].value_counts().rename("records").to_frame())

## Step 8 - Validate the processed file

These checks confirm the column order, unique project IDs, valid years and statuses, and the absence of NRC or phone-number patterns. The final file should contain 488 records: 53 from 2024 and 435 from 2025.

In [ ]:
import re

expected_columns = list(cleaner.OUTPUT_COLUMNS)
assert list(df.columns) == expected_columns
assert len(df) == 488
assert df["project_id"].is_unique
assert (df["project_name"] != "N/A").all()
assert set(df["fiscal_year"]) == {"2024", "2025"}
assert set(df["status"]) <= {"Approved", "Not Approved"}

nrc_pattern = re.compile(r"\b\d{4,7}\s*/\s*\d{1,3}(?:\s*/\s*\d)?\b")
phone_pattern = re.compile(r"\b0?9[567]\d{7}\b")
search_text = df.astype(str).agg(" ".join, axis=1)
assert not search_text.str.contains(nrc_pattern, regex=True).any()
assert not search_text.str.contains(phone_pattern, regex=True).any()

print("All validation checks passed.")
print("Missing ward values:", (df["ward"] == "N/A").sum())
print("Missing sector values:", (df["sector"] == "N/A").sum())

## Step 9 - Check the output format

The final file uses the format agreed for the repository:

- Filename: `db-unza26-csc4792-kalomo_town_council_cdf_projects.csv`
- Location: `data/processed/`
- Delimiter: pipe (`|`)
- Schema: the 10 CDF project columns documented in `docs/DATA_DICTIONARY.md`
- Unique identifier: `project_id`
- Source details: `source_url` and `date_scraped` on every row

The raw OCR file stays in `data/raw/cdf_projects/`. This allows the cleaning script to be rerun without adding the temporary PDF folder to the repository.

## Limitations

- A small number of project names still contain OCR spelling errors. I did not rewrite unclear text by guessing.
- One 2024 disaster-component record covers several health posts and does not identify one clear ward, so its ward is `N/A`.
- One 2025 sector cell is unreadable after OCR, so it remains `N/A`.
- Allocation and disbursement amounts are `N/A` because the source tables do not provide reliable figures for each project.
- This dataset covers community projects only. Bursary beneficiaries need a different schema and include personal information, so they were excluded.
- The temporary source PDFs are not in the repository. Their council pages are recorded in the `source_url` column.

See `docs/DATA_DICTIONARY.md` for the complete column descriptions.